In [1]:
 from pynq import Overlay
 ol = Overlay("Bach_Decoder/vae.bit")

In [2]:
ol.ip_dict

{'axi_intc_0': {'addr_range': 65536,
  'device': <pynq.pl_server.device.XlnkDevice at 0xb442ea50>,
  'driver': pynq.overlay.DefaultIP,
  'fullpath': 'axi_intc_0',
  'gpio': {},
  'interrupts': {'intr': {'controller': 'axi_intc_0',
    'fullpath': 'axi_intc_0/intr',
    'index': 0}},
  'mem_id': 's_axi',
  'parameters': {'C_ADDR_WIDTH': '32',
   'C_ASYNC_INTR': '0xFFFFFFFE',
   'C_BASEADDR': '0x41200000',
   'C_CASCADE_MASTER': '0',
   'C_DISABLE_SYNCHRONIZERS': '0',
   'C_ENABLE_ASYNC': '0',
   'C_EN_CASCADE_MODE': '0',
   'C_FAMILY': 'zynq',
   'C_HAS_CIE': '1',
   'C_HAS_FAST': '0',
   'C_HAS_ILR': '0',
   'C_HAS_IPR': '1',
   'C_HAS_IVR': '1',
   'C_HAS_SIE': '1',
   'C_HIGHADDR': '0x4120FFFF',
   'C_INSTANCE': 'conv2d_axi_intc_0_1',
   'C_IRQ_ACTIVE': '0x1',
   'C_IRQ_CONNECTION': '1',
   'C_IRQ_IS_LEVEL': '1',
   'C_IVAR_RESET_VALUE': '0x0000000000000010',
   'C_KIND_OF_EDGE': '0xFFFFFFFF',
   'C_KIND_OF_INTR': '0xfffffffe',
   'C_KIND_OF_LVL': '0xFFFFFFFF',
   'C_MB_CLK_NOT_CONNE

In [3]:
my_ip = ol.vae_model_0

In [4]:
my_ip.register_map

RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, RESERVED_3=0, RESERVED_4=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0, CHAN2_INT_EN=0, RESERVED_1=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0, CHAN2_INT_ST=0, RESERVED_1=0),
  input_r_1 = Register(input_r=0),
  input_r_2 = Register(input_r=0),
  kernel_1_1 = Register(kernel_1=0),
  kernel_1_2 = Register(kernel_1=0),
  kernel_2_1 = Register(kernel_2=0),
  kernel_2_2 = Register(kernel_2=0),
  kernel_3_1 = Register(kernel_3=0),
  kernel_3_2 = Register(kernel_3=0),
  kernel_4_1 = Register(kernel_4=0),
  kernel_4_2 = Register(kernel_4=0),
  bias_1_1 = Register(bias_1=0),
  bias_1_2 = Register(bias_1=0),
  bias_2_1 = Register(bias_2=0),
  bias_2_2 = Register(bias_2=0),
  bias_3_1 = Register(bias_3=0),
  bias_3_2 = Register(bias_3=0),
  bias_4_1 = Register(bias_4=0),
  bias_4_2 = Regi

In [5]:
import fxpmath
SAVE_DIR  = "DATA_DECODER/process"
IMAGE_SIZE = (64, 64)

import os
import glob
import cv2
import numpy as np
from fxpmath import Fxp
from tqdm import tqdm

fxp_ref = Fxp(None, dtype='fxp-s16/14')  # Định nghĩa kiểu fixed-point

def get_subset_fixed_point(pathname, name=""):
    images = list()

    for fn in tqdm(glob.glob(pathname), desc=name):
        # Đọc ảnh và xử lý
        image = cv2.imread(fn, flags=cv2.IMREAD_COLOR)
        if image is None:
            print(f"LỖI: Không thể đọc ảnh {fn}")
            continue  # Bỏ qua ảnh lỗi, tránh lỗi `cvtColor`

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMAGE_SIZE).astype(np.float16) / 255.0  # Chuẩn hóa về [0, 1]

        # Chuyển đổi sang fixed-point
        image_fixed_point = Fxp(image, like=fxp_ref)  # <-- Lùi về đúng mức thụt lề
        images.append(image_fixed_point)  # Lưu giá trị fixed-point

    return np.array(images)

# Gọi hàm xử lý ảnh
x_print_10 = np.load(os.path.join(SAVE_DIR, "x_print_10.npy"), allow_pickle=True)
x_hole_10 = np.load(os.path.join(SAVE_DIR, "x_hole_10.npy"), allow_pickle=True)
x_crack_10 = np.load(os.path.join(SAVE_DIR, "x_crack_10.npy"), allow_pickle=True)
x_cut_10 = np.load(os.path.join(SAVE_DIR, "x_cut_10.npy"), allow_pickle=True)
x_good_10 = np.load(os.path.join(SAVE_DIR, "x_good_10.npy"), allow_pickle=True)


In [6]:
scale_factor = 2**12  # Hệ số nhân để chuyển về int16

x_print_10_int16 = np.array([(img * scale_factor).astype(np.int16) for img in x_print_10])
x_hole_10_int16  = np.array([(img * scale_factor).astype(np.int16) for img in x_hole_10])
x_crack_10_int16 = np.array([(img * scale_factor).astype(np.int16) for img in x_crack_10])
x_cut_10_int16   = np.array([(img * scale_factor).astype(np.int16) for img in x_cut_10])
x_good_10_int16  = np.array([(img * scale_factor).astype(np.int16) for img in x_good_10])


In [7]:
import numpy as np

# Hàm load dữ liệu từ file và chuyển đổi
def load_data(file_path, dtype=np.int16, scale=2**12):
    return (np.loadtxt(file_path) * scale).astype(dtype)

# Nạp input từ file
input_data = load_data('input16_14.txt')

# Nạp trọng số và bias cho các lớp convolution
layer1_weight = load_data('conv2d_weights.txt')
layer1_bias = load_data('conv2d_bias.txt')

layer2_weight = load_data('conv2d_1_weights.txt')
layer2_bias = load_data('conv2d_1_bias.txt')

layer3_weight = load_data('conv2d_2_weights.txt')
layer3_bias = load_data('conv2d_2_bias.txt')

layer4_weight = load_data('conv2d_3_weights.txt')
layer4_bias = load_data('conv2d_3_bias.txt')

# Nạp trọng số và bias cho các lớp Fully Connected (FC)
fc1_weight = load_data('z_mean_weights.txt')
fc1_bias = load_data('z_mean_bias.txt')

fc2_weight = load_data('z_log_var_weights.txt')
fc2_bias = load_data('z_log_var_bias.txt')

# Kiểm tra kích thước dữ liệu đã nạp
print("Input shape:", input_data.shape)
print("Layer 1 weights shape:", layer1_weight.shape, "Bias shape:", layer1_bias.shape)
print("Layer 2 weights shape:", layer2_weight.shape, "Bias shape:", layer2_bias.shape)
print("Layer 3 weights shape:", layer3_weight.shape, "Bias shape:", layer3_bias.shape)
print("Layer 4 weights shape:", layer4_weight.shape, "Bias shape:", layer4_bias.shape)
print("FC1 weights shape:", fc1_weight.shape, "Bias shape:", fc1_bias.shape)
print("FC2 weights shape:", fc2_weight.shape, "Bias shape:", fc2_bias.shape)


Input shape: (12288,)
Layer 1 weights shape: (192,) Bias shape: (16,)
Layer 2 weights shape: (2048,) Bias shape: (32,)
Layer 3 weights shape: (8192,) Bias shape: (64,)
Layer 4 weights shape: (32768,) Bias shape: (128,)
FC1 weights shape: (65536,) Bias shape: (32,)
FC2 weights shape: (65536,) Bias shape: (32,)


In [8]:
# Nạp trọng số và bias cho các lớp Fully Connected (FC) cua transpose
dense_weight = load_data('Bach_Decoder/dense_weight.txt')
dense_bias = load_data('Bach_Decoder/dense_bias.txt')
epsilon = load_data('Bach_Decoder/epsilon.txt')
# Nạp trọng số và bias cho các lớp convolution transpose
layer1_dec_weight = load_data('Bach_Decoder/conv2d_transpose_weight.txt')
layer1_dec_bias = load_data('Bach_Decoder/conv2d_transpose_bias.txt')

layer2_dec_weight = load_data('Bach_Decoder/conv2d_transpose_1_weight.txt')
layer2_dec_bias = load_data('Bach_Decoder/conv2d_transpose_1_bias.txt')

layer3_dec_weight = load_data('Bach_Decoder/conv2d_transpose_2_weight.txt')
layer3_dec_bias = load_data('Bach_Decoder/conv2d_transpose_2_bias.txt')

layer4_dec_weight = load_data('Bach_Decoder/conv2d_transpose_3_weight.txt')
layer4_dec_bias = load_data('Bach_Decoder/conv2d_transpose_3_bias.txt')




In [9]:
import numpy as np
from pynq import Overlay, allocate

# 🟢 Tạo buffer đầu vào
input_buffer = allocate(shape=(64*64*3,), dtype=np.int16)

# 🟢 Tạo buffer cho các lớp convolution
layer1_weight_buffer = allocate(shape=(2*2*3*16,), dtype=np.int16)
layer1_bias_buffer = allocate(shape=(16,), dtype=np.int16)

layer2_weight_buffer = allocate(shape=(2*2*16*32,), dtype=np.int16)
layer2_bias_buffer = allocate(shape=(32,), dtype=np.int16)

layer3_weight_buffer = allocate(shape=(2*2*32*64,), dtype=np.int16)
layer3_bias_buffer = allocate(shape=(64,), dtype=np.int16)

layer4_weight_buffer = allocate(shape=(2*2*64*128,), dtype=np.int16)
layer4_bias_buffer = allocate(shape=(128,), dtype=np.int16)

# 🟢 Thêm buffer cho Fully Connected (FC)
fc1_weight_buffer = allocate(shape=(2048*32,), dtype=np.int16)  # 2048 input -> 32 output
fc1_bias_buffer = allocate(shape=(32,), dtype=np.int16)  # Bias cho 32 output neurons

fc2_weight_buffer = allocate(shape=(2048*32,), dtype=np.int16)  # 2048 input -> 32 output (FC2 giống FC1)
fc2_bias_buffer = allocate(shape=(32,), dtype=np.int16)  # Bias cho 32 output neurons (FC2)

# 📌 In ra kích thước của buffer
print(f"Input buffer size: {input_buffer.shape}")

print(f"Layer 1 weight buffer size: {layer1_weight_buffer.shape}")
print(f"Layer 1 bias buffer size: {layer1_bias_buffer.shape}")

print(f"Layer 2 weight buffer size: {layer2_weight_buffer.shape}")
print(f"Layer 2 bias buffer size: {layer2_bias_buffer.shape}")

print(f"Layer 3 weight buffer size: {layer3_weight_buffer.shape}")
print(f"Layer 3 bias buffer size: {layer3_bias_buffer.shape}")

print(f"Layer 4 weight buffer size: {layer4_weight_buffer.shape}")
print(f"Layer 4 bias buffer size: {layer4_bias_buffer.shape}")

# ✅ In kích thước buffer của các lớp FC
print(f"FC1 weight buffer size: {fc1_weight_buffer.shape}")
print(f"FC1 bias buffer size: {fc1_bias_buffer.shape}")

print(f"FC2 weight buffer size: {fc2_weight_buffer.shape}")
print(f"FC2 bias buffer size: {fc2_bias_buffer.shape}")


Input buffer size: (12288,)
Layer 1 weight buffer size: (192,)
Layer 1 bias buffer size: (16,)
Layer 2 weight buffer size: (2048,)
Layer 2 bias buffer size: (32,)
Layer 3 weight buffer size: (8192,)
Layer 3 bias buffer size: (64,)
Layer 4 weight buffer size: (32768,)
Layer 4 bias buffer size: (128,)
FC1 weight buffer size: (65536,)
FC1 bias buffer size: (32,)
FC2 weight buffer size: (65536,)
FC2 bias buffer size: (32,)


In [10]:
output_buffer = allocate(shape=(64*64*3,), dtype=np.int16)

In [11]:
import numpy as np
from pynq import Overlay, allocate

# 🟢 Thêm buffer cho Fully Connected (FC)
dense_weight_buffer = allocate(shape=(2048*32,), dtype=np.int16) 
dense_bias_buffer = allocate(shape=(2048,), dtype=np.int16)  
epsilon_buffer = allocate(shape=(32,), dtype=np.int16)

# 🟢 Tạo buffer cho các lớp convolution
layer1_dec_weight_buffer = allocate(shape=(2*2*128*64,), dtype=np.int16)
layer1_dec_bias_buffer = allocate(shape=(64,), dtype=np.int16)

layer2_dec_weight_buffer = allocate(shape=(2*2*64*32,), dtype=np.int16)
layer2_dec_bias_buffer = allocate(shape=(32,), dtype=np.int16)

layer3_dec_weight_buffer = allocate(shape=(2*2*32*16,), dtype=np.int16)
layer3_dec_bias_buffer = allocate(shape=(16,), dtype=np.int16)

layer4_dec_weight_buffer = allocate(shape=(2*2*16*3,), dtype=np.int16)
layer4_dec_bias_buffer = allocate(shape=(3,), dtype=np.int16)



print(f"Layer 1 weight buffer size: {layer1_dec_weight_buffer.shape}")
print(f"Layer 1 bias buffer size: {layer1_dec_bias_buffer.shape}")

print(f"Layer 2 weight buffer size: {layer2_dec_weight_buffer.shape}")
print(f"Layer 2 bias buffer size: {layer2_dec_bias_buffer.shape}")

print(f"Layer 3 weight buffer size: {layer3_dec_weight_buffer.shape}")
print(f"Layer 3 bias buffer size: {layer3_dec_bias_buffer.shape}")

print(f"Layer 4 weight buffer size: {layer4_dec_weight_buffer.shape}")
print(f"Layer 4 bias buffer size: {layer4_dec_bias_buffer.shape}")

# ✅ In kích thước buffer của các lớp FC
print(f"Dense weight buffer size: {dense_weight_buffer.shape}")
print(f"Dense bias buffer size: {dense_bias_buffer.shape}")

print(f"Epsilon buffer size: {epsilon_buffer.shape}")



Layer 1 weight buffer size: (32768,)
Layer 1 bias buffer size: (64,)
Layer 2 weight buffer size: (8192,)
Layer 2 bias buffer size: (32,)
Layer 3 weight buffer size: (2048,)
Layer 3 bias buffer size: (16,)
Layer 4 weight buffer size: (192,)
Layer 4 bias buffer size: (3,)
Dense weight buffer size: (65536,)
Dense bias buffer size: (2048,)
Epsilon buffer size: (32,)


In [12]:
# Sao chép input vào input_buffer
np.copyto(input_buffer, input_data.flatten())

# Sao chép kernel vào các buffer tương ứng
np.copyto(layer1_weight_buffer, layer1_weight.flatten())  # Lớp 1
np.copyto(layer1_bias_buffer, layer1_bias.flatten())  # Bias lớp 1

np.copyto(layer2_weight_buffer, layer2_weight.flatten())  # Lớp 2
np.copyto(layer2_bias_buffer, layer2_bias.flatten())  # Bias lớp 2

np.copyto(layer3_weight_buffer, layer3_weight.flatten())  # Lớp 3
np.copyto(layer3_bias_buffer, layer3_bias.flatten())  # Bias lớp 3

np.copyto(layer4_weight_buffer, layer4_weight.flatten())  # Lớp 4
np.copyto(layer4_bias_buffer, layer4_bias.flatten())  # Bias lớp 4

# ✅ Sao chép dữ liệu vào buffer của Fully Connected (FC)
np.copyto(fc1_weight_buffer, fc1_weight.flatten())  # FC1 weight
np.copyto(fc1_bias_buffer, fc1_bias.flatten())  # FC1 bias

np.copyto(fc2_weight_buffer, fc2_weight.flatten())  # FC2 weight
np.copyto(fc2_bias_buffer, fc2_bias.flatten())  # FC2 bias


In [13]:
# ✅ Sao chép dữ liệu vào buffer của Fully Connected (FC)
np.copyto(dense_weight_buffer, dense_weight.flatten())  # FC1 weight
np.copyto(dense_bias_buffer, dense_bias.flatten())  # FC1 bias
np.copyto(epsilon_buffer, epsilon.flatten())
# Sao chép kernel vào các buffer tương ứng
np.copyto(layer1_dec_weight_buffer, layer1_dec_weight.flatten())  # Lớp 1
np.copyto(layer1_dec_bias_buffer, layer1_dec_bias.flatten())  # Bias lớp 1

np.copyto(layer2_dec_weight_buffer, layer2_dec_weight.flatten())  # Lớp 2
np.copyto(layer2_dec_bias_buffer, layer2_dec_bias.flatten())  # Bias lớp 2

np.copyto(layer3_dec_weight_buffer, layer3_dec_weight.flatten())  # Lớp 3
np.copyto(layer3_dec_bias_buffer, layer3_dec_bias.flatten())  # Bias lớp 3

np.copyto(layer4_dec_weight_buffer, layer4_dec_weight.flatten())  # Lớp 4
np.copyto(layer4_dec_bias_buffer, layer4_dec_bias.flatten())  # Bias lớp 4




In [14]:
# Ghi địa chỉ vật lý của các buffer vào các thanh ghi tương ứng của IP
my_ip.write(0x10, input_buffer.physical_address)  # Địa chỉ vật lý của input_buffer
my_ip.write(0x130, output_buffer.physical_address)

# Ghi địa chỉ vật lý của các buffer trọng số và bias cho các lớp CNN
my_ip.write(0x1c, layer1_weight_buffer.physical_address)  # layer1_weight_buffer
my_ip.write(0x4c, layer1_bias_buffer.physical_address)  # layer1_bias_buffer
my_ip.write(0x28, layer2_weight_buffer.physical_address)  # layer2_weight_buffer
my_ip.write(0x58, layer2_bias_buffer.physical_address)  # layer2_bias_buffer
my_ip.write(0x34, layer3_weight_buffer.physical_address)  # layer3_weight_buffer
my_ip.write(0x64, layer3_bias_buffer.physical_address)  # layer3_bias_buffer
my_ip.write(0x40, layer4_weight_buffer.physical_address)  # layer4_weight_buffer
my_ip.write(0x70, layer4_bias_buffer.physical_address)  # layer4_bias_buffer

# Ghi địa chỉ vật lý của buffer trọng số và bias cho hai lớp Fully Connected (FC)
my_ip.write(0x7c, fc1_weight_buffer.physical_address)  # fc1_weight_buffer
my_ip.write(0x94, fc1_bias_buffer.physical_address)  # fc1_bias_buffer
my_ip.write(0x88, fc2_weight_buffer.physical_address)  # fc2_weight_buffer
my_ip.write(0xa0, fc2_bias_buffer.physical_address)  # fc2_bias_buffer


In [15]:
# Ghi địa chỉ vật lý của buffer trọng số và bias cho hai lớp Fully Connected (FC)
my_ip.write(0xac, dense_weight_buffer.physical_address)  # fc1_weight_buffer
my_ip.write(0xb8, dense_bias_buffer.physical_address)  # fc1_bias_buffer
my_ip.write(0x124, epsilon_buffer.physical_address)  # fc2_weight_buffer
# Ghi địa chỉ vật lý của các buffer trọng số và bias cho các lớp CNN
my_ip.write(0xc4, layer1_dec_weight_buffer.physical_address)  # layer1_weight_buffer
my_ip.write(0xd0, layer1_dec_bias_buffer.physical_address)  # layer1_bias_buffer
my_ip.write(0xdc, layer2_dec_weight_buffer.physical_address)  # layer2_weight_buffer
my_ip.write(0xe8, layer2_dec_bias_buffer.physical_address)  # layer2_bias_buffer
my_ip.write(0xf4, layer3_dec_weight_buffer.physical_address)  # layer3_weight_buffer
my_ip.write(0x100, layer3_dec_bias_buffer.physical_address)  # layer3_bias_buffer
my_ip.write(0x10c, layer4_dec_weight_buffer.physical_address)  # layer4_weight_buffer
my_ip.write(0x118, layer4_dec_bias_buffer.physical_address)  # layer4_bias_buffer




In [16]:
import time

start_time = time.time()

my_ip.write(0x00, 0x01)

while True:
    reg = my_ip.read(0x00)
    if reg != 1:
        break

end_time = time.time()
time_handcoded = end_time - start_time

print("HW time: {}s".format(time_handcoded))


HW time: 0.10638117790222168s


In [17]:
my_ip.register_map.CTRL.AP_DONE


0

In [18]:
import numpy as np

q12 = 12
scale_factor = 2**q12  # 2^12 = 4096

# Chia từng phần tử cho 2^q12
output_buffer = output_buffer / scale_factor

# In kết quả từng phần tử
for value in output_buffer:
    print(value)


0.15576171875
0.155029296875
0.154541015625
0.158203125
0.157958984375
0.1552734375
0.154296875
0.151123046875
0.153076171875
0.154541015625
0.1552734375
0.15771484375
0.16015625
0.161865234375
0.15478515625
0.154541015625
0.1533203125
0.151611328125
0.156494140625
0.16015625
0.1474609375
0.1513671875
0.148681640625
0.148193359375
0.150146484375
0.1474609375
0.146484375
0.15185546875
0.154296875
0.1533203125
0.154296875
0.15234375
0.14892578125
0.147216796875
0.144775390625
0.147705078125
0.138671875
0.142578125
0.144775390625
0.142333984375
0.137451171875
0.13720703125
0.1435546875
0.145751953125
0.147216796875
0.152099609375
0.145751953125
0.146728515625
0.14794921875
0.15185546875
0.149658203125
0.15478515625
0.1513671875
0.15283203125
0.153564453125
0.15087890625
0.147216796875
0.149169921875
0.147705078125
0.14990234375
0.151123046875
0.155029296875
0.1474609375
0.14697265625
0.150390625
0.15234375
0.153564453125
0.154541015625
0.157470703125
0.155517578125
0.14990234375
0.1503906

0.14697265625
0.1494140625
0.15234375
0.154541015625
0.153076171875
0.153564453125
0.154296875
0.153076171875
0.150390625
0.150634765625
0.151611328125
0.14990234375
0.151611328125
0.150390625
0.15185546875
0.148193359375
0.146728515625
0.1494140625
0.1513671875
0.15283203125
0.154052734375
0.151123046875
0.15234375
0.14111328125
0.138916015625
0.154296875
0.171630859375
0.20703125
0.2275390625
0.2900390625
0.310302734375
0.38232421875
0.379150390625
0.433837890625
0.42626953125
0.465087890625
0.46533203125
0.5
0.498291015625
0.4892578125
0.470947265625
0.4794921875
0.462890625
0.44140625
0.41650390625
0.3798828125
0.370849609375
0.326904296875
0.294921875
0.250244140625
0.219482421875
0.1865234375
0.176513671875
0.154541015625
0.143798828125
0.135986328125
0.13330078125
0.140625
0.141357421875
0.142822265625
0.14501953125
0.14599609375
0.151123046875
0.14990234375
0.148681640625
0.151123046875
0.1533203125
0.159912109375
0.1591796875
0.15185546875
0.155517578125
0.15087890625
0.151611

0.57275390625
0.5693359375
0.57177734375
0.5654296875
0.563232421875
0.56005859375
0.554443359375
0.55224609375
0.55224609375
0.53857421875
0.522705078125
0.50927734375
0.49365234375
0.4755859375
0.453125
0.44287109375
0.43408203125
0.404296875
0.3740234375
0.3486328125
0.270263671875
0.2236328125
0.1591796875
0.134521484375
0.140869140625
0.14013671875
0.1416015625
0.14453125
0.151611328125
0.146484375
0.15087890625
0.155029296875
0.142822265625
0.14111328125
0.14208984375
0.139404296875
0.14208984375
0.138916015625
0.13330078125
0.1328125
0.154052734375
0.16943359375
0.226806640625
0.256103515625
0.35595703125
0.378173828125
0.416748046875
0.435546875
0.4375
0.45068359375
0.474609375
0.4833984375
0.497802734375
0.510009765625
0.52001953125
0.53125
0.5390625
0.54296875
0.54931640625
0.552734375
0.552734375
0.5576171875
0.564208984375
0.567626953125
0.568359375
0.568115234375
0.56640625
0.566650390625
0.56201171875
0.562744140625
0.55908203125
0.557861328125
0.551513671875
0.54296875
0

0.144775390625
0.131591796875
0.136474609375
0.141357421875
0.150634765625
0.17724609375
0.19140625
0.223388671875
0.25146484375
0.2880859375
0.302978515625
0.323974609375
0.343017578125
0.3466796875
0.360107421875
0.373046875
0.384033203125
0.379150390625
0.388671875
0.395263671875
0.405029296875
0.411376953125
0.422119140625
0.423583984375
0.431640625
0.404541015625
0.4130859375
0.40966796875
0.416259765625
0.412353515625
0.4111328125
0.40625
0.4140625
0.402587890625
0.403076171875
0.376708984375
0.376708984375
0.352294921875
0.339599609375
0.320556640625
0.30517578125
0.25244140625
0.2353515625
0.193115234375
0.165771484375
0.13623046875
0.122802734375
0.130126953125
0.14013671875
0.134765625
0.132568359375
0.140869140625
0.14697265625
0.151123046875
0.148193359375
0.153076171875
0.150146484375
0.15771484375
0.15771484375
0.15625
0.15625
0.150390625
0.150146484375
0.147216796875
0.148681640625
0.14013671875
0.138671875
0.14306640625
0.143798828125
0.167724609375
0.186767578125
0.216

0.154052734375
0.1484375
0.14111328125
0.141357421875
0.142333984375
0.142333984375
0.148681640625
0.14697265625
0.143310546875
0.146728515625
0.150634765625
0.14990234375
0.15185546875
0.15234375
0.158203125
0.159912109375
0.16357421875
0.159423828125
0.166259765625
0.166015625
0.163330078125
0.16064453125
0.15771484375
0.155029296875
0.152587890625
0.146484375
0.14892578125
0.145751953125
0.148193359375
0.151123046875
0.14990234375
0.1474609375
0.14990234375
0.153076171875
0.141357421875
0.140625
0.142333984375
0.1435546875
0.13623046875
0.134765625
0.1337890625
0.134521484375
0.1328125
0.134765625
0.131591796875
0.1298828125
0.12890625
0.129150390625
0.136474609375
0.141845703125
0.136474609375
0.138671875
0.13525390625
0.1318359375
0.132568359375
0.13330078125
0.135986328125
0.137451171875
0.130859375
0.13037109375
0.132080078125
0.137451171875
0.14501953125
0.141845703125
0.14306640625
0.1435546875
0.140625
0.143798828125
0.143798828125
0.141845703125
0.143310546875
0.147705078125

...

0.148193359375
0.14404296875
0.1474609375
0.150390625
0.1494140625
0.151123046875
0.13818359375
0.136474609375
0.13818359375
0.13720703125
0.13525390625
0.134765625
0.13427734375
0.132568359375
0.135009765625
0.1337890625
0.131591796875
0.13037109375
0.1259765625
0.126953125
0.131103515625
0.133544921875
0.130615234375
0.130126953125
0.130615234375
0.12841796875
0.13037109375
0.130859375
0.131103515625
0.13037109375
0.13330078125
0.132568359375
0.13525390625
0.13671875
0.139892578125
0.1435546875
0.146728515625
0.142333984375
0.140869140625
0.142333984375
0.143310546875
0.143798828125
0.146484375
0.1455078125
0.144287109375
0.146240234375
0.1494140625
0.149169921875
0.150146484375
0.150390625
0.15087890625
0.152587890625
0.154296875
0.15087890625
0.156982421875
0.159423828125
0.15478515625
0.156982421875
0.155029296875
0.15576171875
0.14501953125
0.14306640625
0.149658203125
0.149658203125
0.15087890625
0.146728515625
0.1494140625
0.152099609375
0.14990234375
0.150634765625
0.141845703

0.160888671875
0.1611328125
0.16357421875
0.16796875
0.1650390625
0.166748046875
0.167724609375
0.16845703125
0.169189453125
0.1650390625
0.167724609375
0.172119140625
0.171142578125
0.171630859375
0.169677734375
0.169189453125
0.16796875
0.166015625
0.166015625
0.1669921875
0.167724609375
0.1689453125
0.166748046875
0.16943359375
0.168212890625
0.167236328125
0.16796875
0.169921875
0.168701171875
0.17041015625
0.169189453125
0.154052734375
0.156494140625
0.1552734375
0.15576171875
0.154296875
0.16064453125
0.166259765625
0.167236328125
0.171142578125
0.1748046875
0.171142578125
0.178955078125
0.17919921875
0.181396484375
0.17822265625
0.182861328125
0.18798828125
0.181884765625
0.17919921875
0.1806640625
0.18017578125
0.179443359375
0.172607421875
0.172119140625
0.1708984375
0.1669921875
0.160400390625
0.161865234375
0.15771484375
0.158447265625
0.166748046875
0.166748046875
0.161376953125
0.15966796875
0.162841796875
0.162109375
0.164794921875
0.166748046875
0.165771484375
0.16455078

0.264404296875
0.267578125
0.26611328125
0.264404296875
0.264892578125
0.270263671875
0.26904296875
0.267578125
0.272705078125
0.271240234375
0.26806640625
0.270263671875
0.262939453125
0.267578125
0.26611328125
0.26220703125
0.259033203125
0.259033203125
0.2509765625
0.247314453125
0.238037109375
0.23876953125
0.235107421875
0.22607421875
0.218994140625
0.20654296875
0.197998046875
0.1884765625
0.1728515625
0.1572265625
0.16064453125
0.15625
0.160400390625
0.16259765625
0.158203125
0.159423828125
0.16796875
0.166748046875
0.16943359375
0.16357421875
0.1630859375
0.160888671875
0.1611328125
0.158935546875
0.15576171875
0.157958984375
0.156005859375
0.156005859375
0.14990234375
0.161376953125
0.16796875
0.183837890625
0.21142578125
0.215576171875
0.21728515625
0.228515625
0.239990234375
0.240478515625
0.24755859375
0.24853515625
0.255859375
0.25439453125
0.259033203125
0.25830078125
0.264404296875
0.2646484375
0.265625
0.264892578125
0.268310546875
0.265380859375
0.269775390625
0.266601

0.162109375
0.164306640625
0.1650390625
0.16455078125
0.16455078125
0.1640625
0.1611328125
0.161376953125
0.161865234375
0.156005859375
0.1572265625
0.15673828125
0.170654296875
0.169921875
0.18212890625
0.187744140625
0.19970703125
0.19580078125
0.200439453125
0.2060546875
0.20556640625
0.208984375
0.213623046875
0.21728515625
0.20947265625
0.211181640625
0.21044921875
0.21435546875
0.215576171875
0.215576171875
0.2158203125
0.220947265625
0.21826171875
0.2119140625
0.21240234375
0.216064453125
0.213134765625
0.216796875
0.21484375
0.218994140625
0.214111328125
0.211669921875
0.205322265625
0.210693359375
0.204833984375
0.2041015625
0.1962890625
0.200439453125
0.18896484375
0.18798828125
0.17919921875
0.182861328125
0.15869140625
0.15283203125
0.147705078125
0.146728515625
0.140380859375
0.154296875
0.153564453125
0.149169921875
0.154541015625
0.16015625
0.16015625
0.1650390625
0.159423828125
0.168701171875
0.167724609375
0.164794921875
0.163818359375
0.160400390625
0.159912109375
0.1

0.148681640625
0.150146484375
0.153076171875
0.154541015625
0.15478515625
0.15283203125
0.155517578125
0.156494140625
0.162841796875
0.163330078125
0.15869140625
0.158447265625
0.159423828125
0.158447265625
0.163818359375
0.16650390625
0.1611328125
0.163818359375
0.167236328125
0.16943359375
0.16748046875
0.166259765625
0.16552734375
0.168212890625
0.1650390625
0.1689453125
0.171142578125
0.1728515625
0.172119140625
0.17236328125
0.169189453125
0.16796875
0.165771484375
0.163818359375
0.166259765625
0.16650390625
0.1640625
0.16748046875
0.16748046875
0.165283203125
0.16650390625
0.16552734375
0.155517578125
0.153564453125
0.15625
0.156005859375
0.157470703125
0.15185546875
0.154541015625
0.15673828125
0.150634765625
0.148681640625
0.14599609375
0.154052734375
0.151123046875
0.146484375
0.1513671875
0.15087890625
0.14453125
0.14404296875
0.14599609375
0.147216796875
0.151123046875
0.151123046875
0.152099609375
0.148681640625
0.154052734375
0.152587890625
0.150390625
0.15576171875
0.1579

In [19]:
import numpy as np
np.set_printoptions(threshold=np.inf)  # Hiển thị toàn bộ mảng
print(output_buffer)


[0.15576172 0.1550293  0.15454102 0.15820312 0.15795898 0.15527344
 0.15429688 0.15112305 0.15307617 0.15454102 0.15527344 0.15771484
 0.16015625 0.16186523 0.15478516 0.15454102 0.15332031 0.15161133
 0.15649414 0.16015625 0.14746094 0.15136719 0.14868164 0.14819336
 0.15014648 0.14746094 0.14648438 0.15185547 0.15429688 0.15332031
 0.15429688 0.15234375 0.14892578 0.1472168  0.14477539 0.14770508
 0.13867188 0.14257812 0.14477539 0.14233398 0.13745117 0.13720703
 0.14355469 0.14575195 0.1472168  0.15209961 0.14575195 0.14672852
 0.14794922 0.15185547 0.1496582  0.15478516 0.15136719 0.15283203
 0.15356445 0.15087891 0.1472168  0.14916992 0.14770508 0.14990234
 0.15112305 0.1550293  0.14746094 0.14697266 0.15039062 0.15234375
 0.15356445 0.15454102 0.1574707  0.15551758 0.14990234 0.15039062
 0.1496582  0.15039062 0.15136719 0.15380859 0.1574707  0.15844727
 0.15405273 0.1574707  0.14868164 0.14990234 0.15209961 0.15258789
 0.14770508 0.14526367 0.14379883 0.14550781 0.14111328 0.1437

In [20]:
output_buffer = output_buffer.reshape((3, 64, 64)) 

In [ ]:
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
# Đọc ảnh gốc
image = x_hole_10[3]
# Giả sử output_4 là đầu ra của một lớp Conv2DTranspose có kích thước (num_filters, height, width)
num_filters, height, width = output_buffer.shape
print("Number of filters (channels):", num_filters)
print("Height:", height)
print("Width:", width)

# Đảm bảo có đủ ít nhất 3 kênh, nếu không, sao chép kênh để tạo ảnh RGB
if num_filters >= 3:
    rgb_image = output_buffer[:3]  # Chọn ba kênh đầu tiên
    print("PASS")
else:
    rgb_image = np.zeros((3, height, width), dtype=np.float16)
    for i in range(num_filters):
        rgb_image[i] = output_buffer[i]
    for i in range(num_filters, 3):
        rgb_image[i] = output_buffer[0]  # Sao chép kênh đầu tiên để làm đầy ảnh RGB

# Chuẩn hóa pixel về [0,1]
rgb_image = np.clip(rgb_image, 0, 1)  
rgb_image = (rgb_image * 255).astype(np.uint8)  # Chuyển về 0-255

# Chuyển đổi từ (channels, height, width) sang (height, width, channels)
rgb_image = rgb_image.transpose(1, 2, 0)

# Kiểm tra dữ liệu
print(f"Data type: {rgb_image.dtype}, Shape: {rgb_image.shape}")
print(f"Pixel range: {rgb_image.min()} - {rgb_image.max()}")

# Lưu ảnh
image_recon = Image.fromarray(rgb_image)
image_recon.save("output_image.png")

# Hiển thị ảnh gốc và ảnh tái tạo
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.title("INPUT_DATA")
plt.imshow(image)  # Ảnh gốc
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("OUTPUT_DATA")
plt.imshow(rgb_image)  # Ảnh tái tạo từ mạng
plt.axis("off")

# Chia lại ảnh recon cho 255 trước khi so sánh
rgb_image = rgb_image.astype(np.float16) / 255.0



# Tính toán bản đồ lỗi
error_image = (image - rgb_image) ** 2

# Ngưỡng lỗi
threshold = 0.05
error_image[error_image < threshold] = 0

# Làm đậm lỗi
factor = 1
error_image_enhanced = np.where(error_image > 0, np.clip(error_image * factor, 0, 1), error_image)

# Nếu ảnh có nhiều kênh (RGB), tính toán lỗi trung bình
if error_image.ndim == 3:
    error_image_enhanced = np.mean(error_image_enhanced, axis=-1)

plt.subplot(1, 3, 3)
plt.title("ANOMALY DETECTION")
plt.imshow(error_image_enhanced, cmap="gray")  # Dùng cmap để làm nổi bật lỗi
plt.axis("off")

plt.show()


Number of filters (channels): 3
Height: 64
Width: 64
PASS
Data type: uint8, Shape: (64, 64, 3)
Pixel range: 28 - 149
